# Module 7 – Lab 1: Build a Simple RAG Application using LangChain

**Module:** 7 – Basic LLM Systems (RAG) &nbsp;|&nbsp; **Scenario:** IT Operations Knowledge Assistant &nbsp;|&nbsp; **Knowledge base:** `operations_knowledge_base.csv` (or your own PDF)

In this lab you build a small **Retrieval-Augmented Generation (RAG)** application. An operations engineer asks a question in plain English, and the application answers **only from the operations knowledge base**: server CPU and memory procedures, backup failures, network troubleshooting and incident escalation.

**Learning objectives**

1. Load a CSV or PDF knowledge base with LangChain loaders.
2. Split documents into chunks.
3. Generate embeddings with OpenAI.
4. Store embeddings in a local FAISS vector store and retrieve relevant chunks.
5. Generate a grounded answer with a simple LCEL RAG chain.

Run the cells **in order from top to bottom**. Each section explains the concept before the code that demonstrates it.

## Section 1 – Introduction to RAG

**What is RAG?** Retrieval-Augmented Generation means the application first *retrieves* relevant text from a knowledge base and then asks an LLM to *generate* an answer using that text.

**Why do LLMs need external knowledge?** An LLM knows only what was in its training data, up to a cut-off date. It has never read your company's runbooks, thresholds or escalation contacts. If you ask it about them, it may give a plausible-sounding but wrong answer.

| | Normal LLM question | RAG-based question |
|---|---|---|
| Source of facts | The model's training data | Passages retrieved from your documents |
| Company-specific procedures | Often missing or invented | Taken from the supplied knowledge base |
| Can it cite where the answer came from? | No | Yes, the retrieved records can be shown |

**Basic RAG architecture (the flow used in this lab):**

```mermaid
flowchart LR
    Q[User Question] --> R[Retriever]
    KB[(Operations Knowledge Base)] --> R
    R --> D[Relevant Documents]
    D --> L[LLM]
    Q --> L
    L --> A[Answer]
```

**The five RAG stages you will build**

1. **Document Loading**: read the CSV or PDF file into LangChain `Document` objects.
2. **Text Splitting**: cut long documents into smaller chunks.
3. **Embedding Generation**: convert each chunk into a numerical vector that captures its meaning.
4. **Document Retrieval**: store the vectors and find the chunks closest to the user's question.
5. **Response Generation**: give the retrieved chunks and the question to an LLM, which writes the answer.

**Role of the knowledge base and embeddings:** the knowledge base is the only source of facts the assistant may use. Embeddings are what make the search *semantic*: a question about "high processor usage" can find a record titled "Server CPU utilization above 90 percent" even though the words differ.

## Section 2 – Environment Setup

**Required packages.** The file `requirements.txt` lists the packages for this lab. The core is `langchain==1.3.14`, the same LangChain version used in the previous Module 7 Lab 1. The related packages (`langchain-core`, `langchain-community`, `langchain-openai` and `langchain-text-splitters`) are pinned to the versions that work with it. The other packages are `faiss-cpu` (local vector store), `pypdf` (PDF loading) and `python-dotenv` (reads the API key).

**Existing Python environment.** This lab uses the Python interpreter that is already installed on your machine. Do **not** create a new virtual environment or install another Python. If a package is missing, install it into this same interpreter with:

```
python -m pip install -r requirements.txt
```

**OpenAI API configuration.** Copy `.env.example` to a file named `.env` in this folder and replace the placeholder with your key. The notebook loads the key with `python-dotenv`. The key is never written into the notebook and is never printed.

The LangChain model objects (embeddings and chat model) are created where they are first used, in Sections 5 and 7.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader, PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All libraries imported.")

In [2]:
# Read OPENAI_API_KEY from the .env file in this folder
load_dotenv()

# Lab settings: change these values to experiment
CHAT_MODEL = "gpt-4o-mini"                  # the LLM that writes the answer
EMBEDDING_MODEL = "text-embedding-3-small"  # the model that turns text into vectors
KB_PATH = "operations_knowledge_base.csv"   # knowledge base file: .csv (default) or .pdf
CHUNK_SIZE = 800                            # maximum characters per chunk
CHUNK_OVERLAP = 100                         # characters shared by neighbouring chunks
TOP_K = 3                                   # number of chunks retrieved per question

In [3]:
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-core", "langchain-community",
                "langchain-openai", "langchain-text-splitters", "faiss-cpu", "pypdf"]:
    print(f"{package:26} {version(package)}")

api_key_found = bool(os.getenv("OPENAI_API_KEY"))
print("\nOPENAI_API_KEY loaded:", api_key_found)  # the key itself is never printed
if not api_key_found:
    raise EnvironmentError("OPENAI_API_KEY not found. Copy .env.example to .env and add your key.")

Python: 3.10.11
langchain                  1.3.14
langchain-core             1.5.3
langchain-community        0.4.2
langchain-openai           1.4.2
langchain-text-splitters   1.1.2
faiss-cpu                  1.14.3
pypdf                      6.14.2

OPENAI_API_KEY loaded: True


## Section 3 – Load the Operations Knowledge Base

**Why load documents first?** A RAG application can only use text it has actually read. A *loader* opens a file and converts its content into LangChain `Document` objects. Every `Document` has two parts:

- `page_content`: the text itself.
- `metadata`: information about where the text came from, for example the file name and the row number (CSV) or page number (PDF).

**How the loaders work in this lab**

- **CSV (default):** `CSVLoader` creates one `Document` per row. Each column name and value is written on its own line, such as `Title: Server CPU utilization above 90 percent`.
- **PDF (optional):** `PyPDFLoader` creates one `Document` per page of the PDF.

The loader is chosen from the file extension, so you only change `KB_PATH` in the configuration cell to switch between the CSV and your own PDF. The README explains the steps.

In [4]:
def get_loader(file_path):
    """Return the LangChain loader that matches the file extension."""
    extension = Path(file_path).suffix.lower()
    if extension not in (".csv", ".pdf"):
        raise ValueError(f"Unsupported file type '{extension}'. Use a .csv or .pdf file.")
    if not Path(file_path).exists():
        raise FileNotFoundError(
            f"Knowledge base file '{file_path}' was not found. Check KB_PATH in the configuration cell "
            "and make sure the file is in the same folder as this notebook."
        )
    if extension == ".csv":
        # utf-8-sig also reads CSV files saved by Excel, which may add a hidden BOM
        return CSVLoader(file_path=file_path, encoding="utf-8-sig")
    return PyPDFLoader(file_path)

loader = get_loader(KB_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents from {KB_PATH}")

Loaded 29 documents from operations_knowledge_base.csv


In [5]:
print("Number of documents loaded:", len(documents))

print("\n--- Sample document: page_content ---")
print(documents[0].page_content)

print("\n--- Sample document: metadata ---")
print(documents[0].metadata)

Number of documents loaded: 29

--- Sample document: page_content ---
Record_ID: OPS-001
Category: Server
Title: Server CPU utilization above 90 percent
Description: Sustained CPU utilization above 90 percent slows application response and can delay user transactions. Short spikes during scheduled batch jobs are expected. Utilization above 90 percent for more than 10 minutes must be investigated.
Procedure: 1) Confirm the CPU reading on the monitoring dashboard and check that it has lasted at least 10 minutes. 2) Identify the top CPU consuming processes. 3) Check whether a backup, batch job or patch installation is running and pause non-critical jobs. 4) If a runaway process is found, restart only that process after confirming with the application owner. 5) Record CPU readings and actions in the incident ticket. 6) If CPU remains above 90 percent for 30 minutes, escalate.
Severity: High
Escalation: L2 Server Operations on-call

--- Sample document: metadata ---
{'source': 'operations_k

## Section 4 – Text Splitting

**Why split text?** Embedding models and LLM prompts work best with short, focused passages. If a whole PDF page is treated as one unit, a search returns many unrelated topics at once. Splitting makes each unit small enough to match one specific question.

**Key terms**

- **Chunk:** a piece of text cut from a document.
- **Chunk size:** the maximum number of characters in one chunk. Small chunks match precisely but can lose context. Large chunks keep context but dilute the match.
- **Chunk overlap:** characters repeated at the end of one chunk and the start of the next. This stops a sentence that is cut at a boundary from being lost completely.

`RecursiveCharacterTextSplitter` tries to split at paragraph breaks first, then line breaks, then spaces. With `CHUNK_SIZE = 800` and `CHUNK_OVERLAP = 100`, most operations records (about 700 characters each) stay as a single chunk, and only the longer records are split.

**How splitting improves retrieval:** the search returns the passage that answers the question, not the whole document.

**Watch for the trade-off.** When a long record is split, the second chunk does not repeat the `Record_ID` or `Title`. In the output of the next cells, *Chunk 1* starts at the `Procedure:` line. A chunk that is matched on its own can therefore lose its context. This is why chunk size is a design choice you can test in Exercise 2.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = text_splitter.split_documents(documents)

if not chunks:
    raise ValueError(
        "No text was found in the knowledge base. If it is a PDF, check that it has selectable text. "
        "A scanned PDF is only an image and must be converted with OCR first."
    )

In [7]:
print(f"{len(documents)} documents were split into {len(chunks)} chunks.")

for index in range(2):
    print(f"\n--- Chunk {index} | metadata: {chunks[index].metadata} ---")
    print(chunks[index].page_content)

29 documents were split into 34 chunks.

--- Chunk 0 | metadata: {'source': 'operations_knowledge_base.csv', 'row': 0} ---
Record_ID: OPS-001
Category: Server
Title: Server CPU utilization above 90 percent
Description: Sustained CPU utilization above 90 percent slows application response and can delay user transactions. Short spikes during scheduled batch jobs are expected. Utilization above 90 percent for more than 10 minutes must be investigated.

--- Chunk 1 | metadata: {'source': 'operations_knowledge_base.csv', 'row': 0} ---
Procedure: 1) Confirm the CPU reading on the monitoring dashboard and check that it has lasted at least 10 minutes. 2) Identify the top CPU consuming processes. 3) Check whether a backup, batch job or patch installation is running and pause non-critical jobs. 4) If a runaway process is found, restart only that process after confirming with the application owner. 5) Record CPU readings and actions in the incident ticket. 6) If CPU remains above 90 percent for 3

## Section 5 – Embedding Generation

**What is an embedding?** An embedding is a list of numbers (a *vector*) that represents the meaning of a piece of text. The embedding model reads the text and places it at a point in a high-dimensional space. Texts with similar meaning are placed close together, even when they use different words.

**Why semantic similarity matters:** a keyword search for "processor usage" finds nothing if the record says "CPU". An embedding search understands that both describe the same problem.

**What the generated vector represents:** the model used here, `text-embedding-3-small`, returns a vector with 1,536 numbers. No single number has a human-readable meaning. Together they describe where the text sits in "meaning space". The retriever later compares the question's vector with each chunk's vector and picks the closest ones.

**How embeddings support retrieval:** every chunk is converted into a vector once, when the vector store is built. At question time, the question is converted into a vector in the same way, and the closest chunk vectors are returned.

We use `OpenAIEmbeddings` from LangChain, so the rest of the code does not depend on the OpenAI API details.

In [8]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
print("Embedding model ready:", EMBEDDING_MODEL)

Embedding model ready: text-embedding-3-small


In [9]:
sample_text = "High CPU utilization on an application server"
sample_vector = embeddings.embed_query(sample_text)

print("Sample text:", sample_text)
print("Embedding dimensions:", len(sample_vector))
print("First 5 values:", [round(value, 4) for value in sample_vector[:5]])

Sample text: High CPU utilization on an application server
Embedding dimensions: 1536
First 5 values: [-0.0356, 0.0604, 0.0813, 0.0045, 0.0255]


## Section 6 – Vector Store and Retrieval

**What does a vector store do?** It keeps each chunk's text together with its embedding, so the application can search the chunks by meaning.

**Why FAISS?** FAISS is a local vector index that runs inside this notebook. It needs no server, no account and no hosted database. The index lives in memory and is rebuilt each time the notebook runs.

**What is similarity search?** The question is converted into a vector, compared with every chunk vector, and the `k` closest chunks are returned. Here `TOP_K = 3`, so three chunks come back for each question.

**How a question is matched:** the question and the chunks are both expressed as vectors. The chunks whose vectors point in the most similar direction to the question are the most relevant.

In [10]:
vector_store = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store created with", vector_store.index.ntotal, "vectors.")

FAISS vector store created with 34 vectors.


In [11]:
sample_question = "What should be checked when CPU utilization exceeds 90%?"
retrieved_docs = vector_store.similarity_search(sample_question, k=TOP_K)

print("Question:", sample_question)
print("Number of relevant documents returned:", len(retrieved_docs))

for doc in retrieved_docs:
    print(f"\n--- metadata: {doc.metadata} ---")
    print(doc.page_content[:250], "...")

Question: What should be checked when CPU utilization exceeds 90%?
Number of relevant documents returned: 3

--- metadata: {'source': 'operations_knowledge_base.csv', 'row': 0} ---
Procedure: 1) Confirm the CPU reading on the monitoring dashboard and check that it has lasted at least 10 minutes. 2) Identify the top CPU consuming processes. 3) Check whether a backup, batch job or patch installation is running and pause non-criti ...

--- metadata: {'source': 'operations_knowledge_base.csv', 'row': 0} ---
Record_ID: OPS-001
Category: Server
Title: Server CPU utilization above 90 percent
Description: Sustained CPU utilization above 90 percent slows application response and can delay user transactions. Short spikes during scheduled batch jobs are expect ...

--- metadata: {'source': 'operations_knowledge_base.csv', 'row': 1} ---
Record_ID: OPS-002
Category: Server
Title: High memory utilization
Description: Memory utilization above 90 percent can cause swapping, slow applications and out-o

## Section 7 – Build the Simple RAG Chain

**How retrieval and generation work together:**

1. The **retriever** finds the top `TOP_K` chunks for the question.
2. The chunks are formatted into a **context** block.
3. A **prompt** combines the context, the question and clear instructions.
4. The **LLM** writes the answer from that prompt.
5. An **output parser** turns the model's reply into plain text.

**Why the LLM should answer from the retrieved context:** without it, the model answers from general training, and may describe a generic procedure that does not match your company's runbook. The prompt tells the model to stay inside the supplied context and to say so when the answer is not there.

**What the RAG prompt does:** it has three parts: the retrieved context, the user's question, and the rules (answer only from the context, do not invent steps, and use the exact "not found" sentence when needed).

**The chain pattern (LCEL):** each `|` passes the output of one step to the next:

`{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()`

This is the same LCEL approach used in the previous Module 7 Lab 1 with `langchain==1.3.14`. No older `RetrievalQA` class is needed.

In [12]:
# temperature=0 keeps answers consistent and close to the source text
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
print("Chat model ready:", CHAT_MODEL)

Chat model ready: gpt-4o-mini


In [13]:
RAG_PROMPT_TEXT = """You are an IT Operations Knowledge Assistant.
Answer the question using ONLY the information in the context below.

Rules:
1. If the context contains information that answers the question, give that answer. Use the wording of the context and do not add steps, thresholds or contacts that are not in the context.
2. Only if the context contains no relevant information, reply with exactly this sentence and nothing else (no quotation marks):
The answer could not be found in the available Operations Knowledge Base.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEXT)
print("RAG prompt created. Input variables:", rag_prompt.input_variables)

RAG prompt created. Input variables: ['context', 'question']


In [14]:
def format_docs(docs):
    """Join the retrieved chunks into one block of context text."""
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG chain built. Each question retrieves the top", TOP_K, "chunks.")

RAG chain built. Each question retrieves the top 3 chunks.


In [15]:
question = "What should be checked when CPU utilization exceeds 90%?"
answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: What should be checked when CPU utilization exceeds 90%?

Answer:
Check whether a backup, batch job or patch installation is running and pause non-critical jobs. If a runaway process is found, restart only that process after confirming with the application owner. Record CPU readings and actions in the incident ticket. If CPU remains above 90 percent for 30 minutes, escalate.


## Section 8 – Ask Questions from the Operations Knowledge Base

The function below wraps the chain so you can ask any question with one call. The chain takes the question as a plain string, so the function simply passes it through.

The first five questions are typical operations questions that the knowledge base can answer. The last question is **not** covered by the knowledge base. A correct assistant should say the answer could not be found, rather than invent a procedure.

In [16]:
def ask_question(question):
    """Send one question through the RAG chain and return the answer text."""
    return rag_chain.invoke(question)

In [17]:
ops_questions = [
    "What should be checked when server CPU utilization is above 90%?",
    "How should a backup failure be handled?",
    "What steps should be followed for network connectivity issues?",
    "How should a critical infrastructure incident be escalated?",
    "What should be done when disk utilization is high?",
]

for number, question in enumerate(ops_questions, start=1):
    print(f"Q{number}: {question}")
    print(f"A{number}: {ask_question(question)}")
    print()

Q1: What should be checked when server CPU utilization is above 90%?


A1: 1) Confirm the CPU reading on the monitoring dashboard and check that it has lasted at least 10 minutes. 2) Identify the top CPU consuming processes. 3) Check whether a backup, batch job or patch installation is running and pause non-critical jobs. 4) If a runaway process is found, restart only that process after confirming with the application owner. 5) Record CPU readings and actions in the incident ticket. 6) If CPU remains above 90 percent for 30 minutes, escalate.

Q2: How should a backup failure be handled?


A2: 1) Open the backup job log and note the first error message and its time. 2) Check free space on the backup target. 3) Confirm the backup agent service is running on the source server. 4) Check that the source server is reachable on the backup network. 5) After the cause is fixed, re-run the failed job once outside peak hours. 6) If the job fails again, or any production server has no successful backup in 24 hours, escalate to the Backup Administrator and open an incident.

Q3: What steps should be followed for network connectivity issues?


A3: 1) Confirm the scope: one user, one server, one floor or the whole site. 2) Check the link status and IP address on the affected device, then ping the default gateway. 3) If the gateway does not respond, check the switch port status and VLAN assignment. 4) If the gateway responds, ping the destination by IP address and then by hostname to separate routing and DNS issues. 5) Use traceroute to find the hop where packets stop. 6) If more than one device or a core switch is affected, escalate to the Network Operations Centre.

Q4: How should a critical infrastructure incident be escalated?


A4: Escalate to the IT Operations Manager if service is not restored within 1 hour. Escalate to the IT Director if service is not restored within 4 hours or if data loss is suspected.

Q5: What should be done when disk utilization is high?


A5: When disk utilization is high, follow the high disk utilization procedure: 1) Identify the volume and the largest folders using the disk analysis tool. 2) Check whether log files, temporary files or old backup files are the cause. 3) Compress or move log files older than 30 days to the archive location, as approved by the application owner. 4) Do not delete application data or database files without owner approval. 5) If free space remains below 10 percent after cleanup, raise a storage capacity request.



In [18]:
missing_question = "What is the procedure for replacing a faulty UPS battery in the data centre?"

print("Question (not in the knowledge base):", missing_question)
print("Answer:", ask_question(missing_question))

Question (not in the knowledge base): What is the procedure for replacing a faulty UPS battery in the data centre?


Answer: The answer could not be found in the available Operations Knowledge Base.


## Section 9 – Understanding the RAG Pipeline

Follow one question through every stage. The question travels like this:

**Question → embedding → similarity search over chunk vectors → top-k chunks → prompt with context → LLM → answer**

| Stage | What you inspect | Where it was created |
|---|---|---|
| Original documents | Loaded `Document` objects | Section 3 |
| Text chunks | Smaller pieces of text | Section 4 |
| Embeddings | Vectors of length 1,536 | Sections 5 and 6 |
| Retrieved documents | The chunks closest to the question | Section 6 and this section |
| Final generated answer | The text written by the LLM | Section 7 and this section |

The next two cells trace one question through these stages.

In [19]:
trace_question = "How often should server backups be performed?"

print("1. Original documents loaded:", len(documents))
print("2. Text chunks created:", len(chunks))

question_vector = embeddings.embed_query(trace_question)
print("3. Question embedding length:", len(question_vector))

retrieved = vector_store.similarity_search(trace_question, k=TOP_K)
print("4. Retrieved chunks (first line of each):")
for doc in retrieved:
    print("   -", doc.page_content.splitlines()[0])

1. Original documents loaded: 29
2. Text chunks created: 34


3. Question embedding length: 1536


4. Retrieved chunks (first line of each):
   - Record_ID: OPS-010
   - Record_ID: OPS-012
   - Record_ID: OPS-004


In [20]:
print("5. Final generated answer for:", trace_question)
print(rag_chain.invoke(trace_question))

5. Final generated answer for: How often should server backups be performed?


Full backups run weekly on Sunday at 01:00. Incremental backups run daily at 01:00 from Monday to Saturday.


## Section 10 – Hands-on Exercises and Quiz

### Exercises

**Exercise 1 – Add three new records.** Open `operations_knowledge_base.csv` in Excel or a text editor. Add three rows with new Record IDs (OPS-030, OPS-031, OPS-032) in the same column order. Save the file, then re-run the notebook from Section 3. Check that the document count has gone up by three, and ask a question that one of the new records answers.

**Exercise 2 – Change the chunk size.** In the configuration cell, change `CHUNK_SIZE` to 300, 500 and 1000 (keep `CHUNK_OVERLAP` at 100). Re-run Section 4 and the vector store cells in Section 6. Our test run gave these approximate chunk counts: 300 gives about 122 chunks, 500 about 71, 800 about 34 and 1,000 about 29. Explain why smaller chunks create more of them.

**Exercise 3 – Change top-k from 3 to 5.** Set `TOP_K = 5`, then re-run the retriever cell and the sample question in Section 7. Compare the retrieved chunks. Do the extra two chunks help or add noise?

**Exercise 4 – Make the answers more concise.** Edit `RAG_PROMPT_TEXT` in the prompt cell, for example by adding "Answer in at most three short bullet points." Re-run the prompt, chain and question cells, then re-run the questions in Section 8. Keep the exact "not found" sentence in the prompt.

**Exercise 5 – Ask a question the knowledge base cannot answer.** Call `ask_question` with a question such as "What is the procedure for onboarding a new employee?" Examine the response. Does it say the answer could not be found? If it guesses, strengthen the rules in the prompt (Exercise 4) and try again.

### Quiz (multiple choice)

**Q1. What is the main purpose of RAG?**
- A) Train a new LLM from scratch
- B) Give an LLM relevant external context at the time of the question
- C) Replace the vector store with a database
- D) Compress PDF files

**Q2. Which LangChain loader does this lab use for a CSV knowledge base?**
- A) PyPDFLoader
- B) CSVLoader
- C) RecursiveCharacterTextSplitter
- D) FAISS

**Q3. Why are documents split into chunks before they are embedded?**
- A) To make the LLM answer faster by removing all context
- B) To retrieve only the passages that are relevant to a question
- C) To convert PDF files into CSV files
- D) To use the document title as the database key

**Q4. What does an embedding represent?**
- A) The password used to call the API
- B) The file size of the document
- C) A list of numbers that captures the meaning of a piece of text
- D) The page number of a PDF

**Q5. Which component decides which chunks are passed to the LLM for a question?**
- A) The LLM itself
- B) The retriever (similarity search with top-k)
- C) The StrOutputParser
- D) The CSV loader

---

## Instructor Answer Key (not for participants)

| Question | Correct answer | Short explanation |
|---|---|---|
| Q1 – RAG fundamentals | **B** | RAG supplies relevant external text to the LLM when the question is asked. The model is not retrained. |
| Q2 – Document loading | **B** | `CSVLoader` reads each CSV row as a `Document`. `PyPDFLoader` is for PDFs. |
| Q3 – Text splitting | **B** | Smaller chunks let the search return the specific passage that matches the question. |
| Q4 – Embeddings | **C** | An embedding is a numeric vector. Similar meanings give vectors that are close together. |
| Q5 – Retrieval and generation | **B** | The retriever runs a similarity search and returns the top-k chunks, which become the context for the LLM. |

**Facilitator notes**

- If participants see different chunk counts, check `CHUNK_SIZE` and `CHUNK_OVERLAP` first.
- Answers can vary slightly between runs. Check that the facts match the cited record, not the exact wording.
- For the unanswerable question in Section 8, the expected behaviour is the exact sentence "The answer could not be found in the available Operations Knowledge Base." Any invented procedure is a failure to discuss with the group.